In [1]:
import torch
import torch.nn as nn
import math

In [ ]:
# input -> embedding + postional 

In [39]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model:int, vocab_size:int, padding_idx:int=None):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size,d_model, padding_idx=padding_idx)
    def forward(self,x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [40]:
# padding_idx = 0 thì không cập nhật trọng số của từ có chỉ số 0 trong quá trình huấn luyện
input_emb = InputEmbeddings(2,10,padding_idx=0) # tạo bảng tra cứu với kích thước 2 và vocab size là 10
input_emb.state_dict()

OrderedDict([('embedding.weight',
              tensor([[ 0.0000,  0.0000],
                      [ 0.1805,  1.4355],
                      [-0.2430,  0.4609],
                      [ 0.6545,  0.9839],
                      [ 0.4101,  1.6336],
                      [ 2.0034,  0.3202],
                      [-0.0862, -0.3242],
                      [-0.3235,  0.2734],
                      [-1.0871, -1.1960],
                      [-1.0180, -0.3586]]))])

In [41]:
# input_emb = InputEmbeddings(2,10)

x = torch.LongTensor([[1,2,3],[4,3,2]])
output = input_emb(x)
output

tensor([[[ 0.2553,  2.0301],
         [-0.3437,  0.6518],
         [ 0.9256,  1.3915]],

        [[ 0.5799,  2.3102],
         [ 0.9256,  1.3915],
         [-0.3437,  0.6518]]], grad_fn=<MulBackward0>)

In [31]:
# x.shape = (batch_size, seq_len, d_model)
# ( 2, 3, 2 )
output.shape

torch.Size([2, 3, 2])

In [51]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model:int, seq_len:int, dropout:float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0,seq_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(position*div_term)
        pe[:,1::2] = torch.cos(position*div_term)
        pe = pe.unsqueeze(0)  # shape (1, seq_len, d_model) 
        self.register_buffer('pe',pe)
    def forward(self,x):
        x = x + (self.pe[:,:x.shape[1],:]).requires_grad_(False)
        return self.dropout(x)

In [52]:
pos_enc = PositionalEncoding(d_model=2, seq_len=3, dropout=0.1)
pos_enc.state_dict()

OrderedDict([('pe',
              tensor([[[ 0.0000,  1.0000],
                       [ 0.8415,  0.5403],
                       [ 0.9093, -0.4161]]]))])

In [48]:
output

tensor([[[0.0000, 3.3667],
         [0.5531, 0.0000],
         [2.0388, 1.0837]],

        [[0.6444, 3.6780],
         [1.9634, 0.0000],
         [0.6285, 0.2618]]], grad_fn=<MulBackward0>)

In [ ]:
# pos_enc.eval()
output = pos_enc(output)
output

tensor([[[ 0.0000,  4.3667],
         [ 1.3946,  0.5403],
         [ 2.9481,  0.6676]],

        [[ 0.6444,  4.6780],
         [ 2.8049,  0.5403],
         [ 1.5378, -0.1543]]], grad_fn=<AddBackward0>)

In [ ]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        assert d_model % h == 0, "d_model must be divisible by h"

        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.attention_scores = None

    @staticmethod
    def attention(query, key, value, mask=None, dropout=None):
        d_k = query.size(-1)

        scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = scores.softmax(dim=-1)

        if dropout is not None:
            attention = dropout(attention)

        return attention @ value, attention

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        query = query.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        key   = key.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        value = value.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)

        x, self.attention_scores = self.attention(
            query, key, value, mask, self.dropout
        )

        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.w_o(x)


In [ ]:
 #Input embedding
#      ↓
# Chia ra nhiều "cặp mắt"
#      ↓
# Mỗi cặp mắt nhìn câu theo 1 kiểu
#      ↓
# Ghép kết quả lại
#      ↓
# Trộn lần cuối
#      ↓
# Contextual embedding

In [55]:
multi = MultiHeadAttentionBlock(d_model=2, h=2, dropout=0.1)
multi.state_dict()

OrderedDict()